# 07 — SIEVE (Educational Reproduction)

AG News's 4 balanced labels make an **exhaustive** version of SIEVE's candidate subindex space tractable, so this notebook builds all reasonable candidates rather than approximating with a partial historical workload, and implements the paper's real cost model end-to-end (not a hand-wavy version): `M_down`, subindex memory `S`, indexed-search cost `C`, brute-force cost `C_bf`, `GreedyRatio` construction under a memory budget, and cost-based serving (smallest subsuming subindex vs. brute-force, whichever the cost model says is cheaper).

This notebook is self-contained: it installs its own deps, loads data, builds embeddings, and writes its own results CSV. Only the methodology, config constants, seed, and corpus/query construction are shared verbatim across all 9 notebooks in this study.


## 1. Install

In [1]:
# Core install cell. Safe to re-run. Each notebook is independently runnable.
!pip install -q datasets sentence-transformers hnswlib scikit-learn pandas numpy tqdm


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Imports

In [2]:
import os, time, json, math
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

np.set_printoptions(suppress=True)
pd.set_option("display.max_columns", 50)

import hnswlib


## 3. Shared configuration

In [3]:
# ============================================================
# SHARED CONFIGURATION — identical across all 9 notebooks.
# Only DEBUG_MODE changes corpus/query size; everything else fixed.
# ============================================================
DEBUG_MODE   = True     # small N for correctness checks; set False for full run

N_CORPUS     = 60_000 if not DEBUG_MODE else 3_000
N_QUERIES    = 1_000  if not DEBUG_MODE else 100
K            = 10
SEED         = 42

EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"   # 384-dim
HNSW_M               = 16
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH       = 100
HNSW_SPACE           = "cosine"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DEBUG_MODE={DEBUG_MODE}  N_CORPUS={N_CORPUS}  N_QUERIES={N_QUERIES}  DEVICE={DEVICE}")


DEBUG_MODE=True  N_CORPUS=3000  N_QUERIES=100  DEVICE=cuda


## 4. Dataset — AG News corpus/query construction (shared, verbatim)

**Synthetic attributes, labeled explicitly:** `synthetic_tag` (uniform(0,1) per corpus doc) is used to carve the 10%/5% sub-selectivity filters out of a single-label 25% filter. `synthetic_range_attr` (uniform int in [0,10000)) has no semantic relationship to the article text and exists only so Notebook 08 can run a range-filtered experiment (AG News's label is categorical, not a range attribute).

In [4]:
# ============================================================
# DATASET — AG News. Hard-coded facts per spec:
#   4 classes, exactly balanced: 0=World, 1=Sports, 2=Business, 3=Sci/Tech
#   train: 120,000 rows (30,000/class). test: 7,600 rows (1,900/class).
# Corpus is drawn from train (subsampled to N_CORPUS), queries from test
# (subsampled to N_QUERIES), using SEED=42 for the subsample. This is why
# natural single-label filters have ~25% selectivity — not an arbitrary
# number, it follows directly from AG News's exact 4-way class balance.
# ============================================================
ds = load_dataset("fancyzhx/ag_news")
assert ds["train"].num_rows == 120_000
assert ds["test"].num_rows == 7_600

rng = np.random.default_rng(SEED)

train_idx_all = np.arange(ds["train"].num_rows)
rng_perm = np.random.default_rng(SEED)
corpus_idx = rng_perm.choice(train_idx_all, size=min(N_CORPUS, len(train_idx_all)), replace=False)
corpus_idx.sort()

test_idx_all = np.arange(ds["test"].num_rows)
rng_perm2 = np.random.default_rng(SEED)
query_idx = rng_perm2.choice(test_idx_all, size=min(N_QUERIES, len(test_idx_all)), replace=False)
query_idx.sort()

corpus_texts  = [ds["train"][int(i)]["text"]  for i in corpus_idx]
corpus_labels = np.array([ds["train"][int(i)]["label"] for i in corpus_idx], dtype=np.int64)

query_texts   = [ds["test"][int(i)]["text"]   for i in query_idx]
query_labels  = np.array([ds["test"][int(i)]["label"] for i in query_idx], dtype=np.int64)

# ------------------------------------------------------------
# SYNTHETIC ATTRIBUTES (explicitly labeled as synthetic; not AG News data).
# synthetic_tag: per-corpus-doc uniform(0,1), used to carve the 10%/5%
#   "synthetic controlled" selectivity filters out of a 25% single-label
#   filter (label==c AND synthetic_tag<0.4 / <0.2).
# synthetic_range_attr: per-corpus-doc uniform integer in [0, 10000), used
#   ONLY by Notebook 08 (UNIFY) to construct a range-filtered ANN
#   experiment, since AG News's label is categorical, not a range
#   attribute. It has NO semantic relationship to the article text —
#   it exists purely to make a range-filtered benchmark possible, mirroring
#   the UNIFY paper's own methodology for attribute-less datasets
#   (SIFT1M/GIST1M: uniform random value in [0, 10000)).
# Both are assigned once here, with a fixed seed, and reused identically
# across every notebook that needs them.
# ------------------------------------------------------------
tag_rng = np.random.default_rng(SEED)
synthetic_tag = tag_rng.uniform(0.0, 1.0, size=len(corpus_idx))

range_rng = np.random.default_rng(SEED)
synthetic_range_attr = range_rng.integers(0, 10_000, size=len(corpus_idx))

corpus_df = pd.DataFrame({
    "corpus_pos": np.arange(len(corpus_idx)),
    "train_idx": corpus_idx,
    "label": corpus_labels,
    "synthetic_tag": synthetic_tag,
    "synthetic_range_attr": synthetic_range_attr,
})
print(corpus_df["label"].value_counts().sort_index())
print(corpus_df.head())


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

label
0    768
1    767
2    704
3    761
Name: count, dtype: int64
   corpus_pos  train_idx  label  synthetic_tag  synthetic_range_attr
0           0         61      2       0.773956                   892
1           1        125      3       0.438878                  7739
2           2        146      3       0.858598                  6545
3           3        182      3       0.697368                  4388
4           4        196      3       0.094177                  4330


## 5. Embedding generation

In [5]:
# ============================================================
# EMBEDDING GENERATION — generated once per notebook, cached in-memory
# (and to /content/*.npy for reuse within the same runtime). Never assume
# a prior notebook already produced these files.
# ============================================================
_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)

def embed(texts, cache_path):
    if os.path.exists(cache_path):
        arr = np.load(cache_path)
        if arr.shape[0] == len(texts):
            return arr
    t0 = time.time()
    arr = _model.encode(
        texts, batch_size=128, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)
    print(f"Embedded {len(texts)} texts in {time.time()-t0:.2f}s")
    np.save(cache_path, arr)
    return arr

corpus_emb = embed(corpus_texts, "/content/corpus_emb.npy" if os.path.isdir("/content") else "corpus_emb.npy")
query_emb  = embed(query_texts,  "/content/query_emb.npy"  if os.path.isdir("/content") else "query_emb.npy")
print(corpus_emb.shape, query_emb.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Embedded 3000 texts in 5.63s


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedded 100 texts in 0.18s
(3000, 384) (100, 384)


## 6. Filter design (§4, shared)

In [6]:
# ============================================================
# FILTER DESIGN (per spec §4) — the concrete, non-ambiguous scheme every
# notebook uses. "synthetic controlled" filters explicitly use
# synthetic_tag (see markdown above); everything else is a natural label
# filter derived from AG News's exact class balance.
# ============================================================
def build_filters(corpus_df):
    """Return dict: filter_name -> (selectivity_target_pct, boolean mask, type)."""
    labels = corpus_df["label"].values
    tag = corpus_df["synthetic_tag"].values
    filters = {}
    filters["sel100_unfiltered"] = (100, np.ones(len(labels), dtype=bool), "natural")
    filters["sel75_label_ne_3"]  = (75,  labels != 3, "natural")
    filters["sel50_label_in_01"] = (50,  np.isin(labels, [0, 1]), "natural")
    # single-class filters (~25% target) — report all 4 classes' actual results,
    # but use class 0 as the canonical "sel25" filter referenced elsewhere.
    for c in range(4):
        filters[f"sel25_label_eq_{c}"] = (25, labels == c, "natural")
    # synthetic controlled sub-selectivity filters, built ON TOP OF the
    # canonical single-class filter (class 0), per spec.
    filters["sel10_label0_tag_lt_0.4"] = (10, (labels == 0) & (tag < 0.4), "synthetic_controlled")
    filters["sel5_label0_tag_lt_0.2"]  = (5,  (labels == 0) & (tag < 0.2), "synthetic_controlled")
    return filters

FILTERS = build_filters(corpus_df)
filter_summary = []
for name, (target_pct, mask, ftype) in FILTERS.items():
    filter_summary.append({
        "filter": name, "target_selectivity_pct": target_pct,
        "total_corpus_size": len(mask),
        "valid_document_count": int(mask.sum()),
        "actual_selectivity_pct": round(100.0 * mask.sum() / len(mask), 3),
        "type": ftype,
    })
filter_summary_df = pd.DataFrame(filter_summary)
print(filter_summary_df)


                    filter  target_selectivity_pct  total_corpus_size  \
0        sel100_unfiltered                     100               3000   
1         sel75_label_ne_3                      75               3000   
2        sel50_label_in_01                      50               3000   
3         sel25_label_eq_0                      25               3000   
4         sel25_label_eq_1                      25               3000   
5         sel25_label_eq_2                      25               3000   
6         sel25_label_eq_3                      25               3000   
7  sel10_label0_tag_lt_0.4                      10               3000   
8   sel5_label0_tag_lt_0.2                       5               3000   

   valid_document_count  actual_selectivity_pct                  type  
0                  3000                 100.000               natural  
1                  2239                  74.633               natural  
2                  1535                  51.167      

## 7. Ground truth + recall (shared)

In [7]:
# ============================================================
# GROUND TRUTH + RECALL — exact cosine top-K over the filter-valid subset,
# computed identically in every notebook that needs it.
# ============================================================
def exact_topk_filtered(query_vecs, corpus_vecs, mask, k):
    """Exact cosine top-k (corpus_vecs assumed L2-normalized) restricted to
    corpus rows where mask is True. Returns (indices, sims), indices are
    positions into the FULL corpus (not the masked subset)."""
    valid_pos = np.nonzero(mask)[0]
    if len(valid_pos) == 0:
        return np.full((len(query_vecs), k), -1, dtype=np.int64), np.zeros((len(query_vecs), k))
    sub = corpus_vecs[valid_pos]                       # (m, d)
    sims = query_vecs @ sub.T                           # (nq, m) cosine sim (both normalized)
    kk = min(k, sub.shape[0])
    top_local = np.argsort(-sims, axis=1)[:, :kk]
    top_global = valid_pos[top_local]
    if kk < k:
        pad_idx = np.full((len(query_vecs), k - kk), -1, dtype=np.int64)
        top_global = np.concatenate([top_global, pad_idx], axis=1)
    top_sims = np.take_along_axis(sims, top_local, axis=1)
    if kk < k:
        top_sims = np.concatenate([top_sims, np.zeros((len(query_vecs), k - kk))], axis=1)
    return top_global, top_sims

def recall_at_k(retrieved_ids, gt_ids, k=K):
    """retrieved_ids, gt_ids: (n_queries, k) int arrays of corpus positions (-1 = missing)."""
    total = 0.0
    for r, g in zip(retrieved_ids, gt_ids):
        gt_set = set(int(x) for x in g if x >= 0)
        if len(gt_set) == 0:
            total += 1.0  # nothing to find, nothing missed
            continue
        r_set = set(int(x) for x in r if x >= 0)
        total += len(r_set & gt_set) / min(k, len(gt_set)) if len(gt_set) < k else len(r_set & gt_set) / k
    return total / len(retrieved_ids)


## 8. Latency / QPS / results utilities (shared)

In [8]:
# ============================================================
# LATENCY / QPS UTILITIES — query execution time only. Install / embedding /
# preprocessing time is reported separately, never folded into query latency.
# ============================================================
def time_queries(fn, n_repeats=1):
    """fn() runs ALL queries once and returns per-query latencies (list of floats, seconds).
    Caller is responsible for making fn() do only the search, not setup."""
    all_lat = []
    for _ in range(n_repeats):
        lat = fn()
        all_lat.extend(lat)
    arr = np.array(all_lat)
    return {
        "mean_latency_ms": float(arr.mean() * 1000),
        "p50_latency_ms": float(np.percentile(arr, 50) * 1000),
        "p95_latency_ms": float(np.percentile(arr, 95) * 1000),
        "qps": float(1.0 / arr.mean()) if arr.mean() > 0 else float("inf"),
    }

def append_result(rows, method, filter_name, selectivity, recall, lat_stats,
                   index_build_time_s, **extra):
    row = {
        "method": method, "filter": filter_name, "selectivity": selectivity,
        "recall_at_10": recall,
        "mean_latency_ms": lat_stats["mean_latency_ms"],
        "p50_latency_ms": lat_stats["p50_latency_ms"],
        "p95_latency_ms": lat_stats["p95_latency_ms"],
        "qps": lat_stats["qps"],
        "index_build_time_s": index_build_time_s,
    }
    row.update(extra)
    rows.append(row)
    return rows

def save_results(rows, method_slug):
    df = pd.DataFrame(rows)
    path = f"results_{method_slug}.csv"
    df.to_csv(path, index=False)
    print(f"Wrote {path} ({len(df)} rows)")
    return df


## Original SIEVE vs. Our Implementation

**(A)** SIEVE (Li, Huang, Ding, Park, Chen, VLDB 2025) builds a workload-tailored *set* of ordinary HNSW subindexes (one per observed filter/filter-combination), plus a mandatory base index, and uses a 3D (memory/time/recall) cost model both to choose which subindexes to build (`GreedyRatio`, submodular optimization under a memory budget) and which subindex serves a given query at runtime.

**(B)** This notebook reproduces the cost model and `GreedyRatio` construction exactly, and reproduces cost-based serving exactly (downscaled `sef`, indexed-vs-brute-force choice per query).

**(C)** What's simplified: the *candidate space*. AG News's filter set in this study (per section 6 above) has at most ~10 candidate filters — tiny compared to the paper's thousands-of-filters YFCC/UQV workloads — so this notebook builds the full candidate space exhaustively rather than sampling a historical query log.

**(D vs E):** SIEVE's real value — workload-driven selection over a large, previously unseen filter space — is **not really exercised** by AG News's 4-label space; this is stated explicitly rather than implying the benchmark stresses SIEVE the way the paper's own experiments do. No paper numbers are copied into this notebook's results CSV.

## 9. Candidate subindex set (base index + one subindex per filter in `FILTERS`)

In [9]:
N = len(corpus_emb)

candidates = {"base": np.ones(N, dtype=bool)}
for name, (_, mask, _) in FILTERS.items():
    candidates[name] = mask

print(f"{len(candidates)} candidate subindexes (base + {len(FILTERS)} filters), "
      f"well within the '~10 candidates' scale this notebook targets.")
for name, mask in candidates.items():
    print(f"  {name:28s} card={int(mask.sum())}")


10 candidate subindexes (base + 9 filters), well within the '~10 candidates' scale this notebook targets.
  base                         card=3000
  sel100_unfiltered            card=3000
  sel75_label_ne_3             card=2239
  sel50_label_in_01            card=1535
  sel25_label_eq_0             card=768
  sel25_label_eq_1             card=767
  sel25_label_eq_2             card=704
  sel25_label_eq_3             card=761
  sel10_label0_tag_lt_0.4      card=292
  sel5_label0_tag_lt_0.2       card=156


## 10. SIEVE cost model (implemented exactly, not hand-waved)

- `M_down(I_h) = M_inf * log(card(h)) / log(N)`
- `S(I_h) = M_down(I_h) * card(h)`
- `C(I_h, sef, f) = log(card(h)) * sef * (card(h)/card(f))^cor`, `cor=1`
- `C_bf(f) = gamma * card(f)`, `gamma` calibrated once by matching costs at `card=1000` (as the paper does).

In [10]:
M_INF = HNSW_M
COR = 1.0  # no correlation modeling needed at this scale, per spec

def m_down(card_h, N=N, M_inf=M_INF):
    if card_h <= 1:
        return M_inf
    return M_inf * math.log(card_h) / math.log(N)

def subindex_memory(card_h, N=N, M_inf=M_INF):
    return m_down(card_h, N, M_inf) * card_h

def indexed_search_cost(card_h, sef, card_f, cor=COR):
    if card_h <= 1 or card_f <= 0:
        return float("inf")
    return math.log(card_h) * sef * (card_h / card_f) ** cor

# Calibrate gamma once: match indexed-search-cost and brute-force cost at
# card=1000, using a representative sef (HNSW_EF_SEARCH) and card_f=card_h=1000.
_card_probe = 1000
_sef_probe = HNSW_EF_SEARCH
_cost_at_probe = indexed_search_cost(_card_probe, _sef_probe, _card_probe)
GAMMA = _cost_at_probe / _card_probe

def brute_force_cost(card_f, gamma=GAMMA):
    return gamma * card_f

print(f"Calibrated gamma={GAMMA:.4f} (matching indexed vs brute-force cost at card=1000)")


Calibrated gamma=0.6908 (matching indexed vs brute-force cost at card=1000)


## 11. GreedyRatio construction (submodular, memory-budgeted)

Budget = 3x the base index size (paper's default). Greedily add the candidate with the highest marginal-benefit-per-memory-byte until the budget is exhausted. 'Benefit' of a candidate subindex is estimated as the total query-cost reduction it offers across the candidate filter set, versus always falling back to the base index or brute force.

In [11]:
base_memory = subindex_memory(N)
BUDGET = 3 * base_memory
print(f"base_memory={base_memory:.1f}  BUDGET={BUDGET:.1f}")

def best_cost_for_filter(built_set, card_f, sef_inf=HNSW_EF_SEARCH):
    """Cheapest option to serve filter f given currently-built subindexes:
    min over (each built subindex h that subsumes f, cost via that subindex)
    vs brute-force. 'Subsumes' is approximated at this scale as: h's mask is
    a superset of f's mask (checked by caller)."""
    return None  # placeholder, real logic lives in greedy_ratio()

def subsumes(mask_h, mask_f):
    return bool(np.all(mask_h[mask_f]))  # every f-valid doc is h-valid

def cost_to_serve(card_h, card_f, sef_inf=HNSW_EF_SEARCH):
    sef_down = max(K, sef_inf * math.log(card_h) / math.log(N)) if card_h > 1 else sef_inf
    return indexed_search_cost(card_h, sef_down, card_f)

def greedy_ratio(candidates, budget):
    built = {"base": candidates["base"]}  # base index is mandatory
    remaining = {k: v for k, v in candidates.items() if k != "base"}
    used_memory = subindex_memory(int(built["base"].sum()))
    selection_log = []

    def total_cost(built_dict):
        total = 0.0
        for fname, fmask in candidates.items():
            if fname == "base":
                continue
            card_f = int(fmask.sum())
            options = []
            for hname, hmask in built_dict.items():
                if subsumes(hmask, fmask):
                    card_h = int(hmask.sum())
                    options.append(cost_to_serve(card_h, card_f))
            options.append(brute_force_cost(card_f))
            total += min(options)
        return total

    cur_cost = total_cost(built)
    while remaining:
        best_name, best_ratio, best_mem, best_new_cost = None, -1.0, None, None
        for name, mask in remaining.items():
            card_h = int(mask.sum())
            mem = subindex_memory(card_h)
            if used_memory + mem > budget:
                continue
            trial_built = dict(built); trial_built[name] = mask
            new_cost = total_cost(trial_built)
            benefit = cur_cost - new_cost
            if mem <= 0:
                continue
            ratio = benefit / mem
            if ratio > best_ratio:
                best_ratio, best_name, best_mem, best_new_cost = ratio, name, mem, new_cost
        if best_name is None or best_ratio <= 0:
            break
        built[best_name] = remaining.pop(best_name)
        used_memory += best_mem
        selection_log.append({"added": best_name, "marginal_ratio": best_ratio,
                               "memory_after": used_memory, "cost_after": best_new_cost})
        cur_cost = best_new_cost
    return built, selection_log

built_subindexes, selection_log = greedy_ratio(candidates, BUDGET)
selection_log_df = pd.DataFrame(selection_log)
print(f"GreedyRatio selected {len(built_subindexes)} / {len(candidates)} subindexes "
      f"(incl. mandatory base) under budget={BUDGET:.1f}")
selection_log_df


base_memory=48000.0  BUDGET=144000.0
GreedyRatio selected 3 / 10 subindexes (incl. mandatory base) under budget=144000.0


,added,marginal_ratio,memory_after,cost_after
0,sel50_label_in_01,0.017246,70504.484623,4927.419592
1,sel75_label_ne_3,0.009549,105019.340176,4597.848686


## 12. Build the selected subindexes (real hnswlib indexes)

In [12]:
import hnswlib
built_indexes = {}
build_times = {}
for name, mask in built_subindexes.items():
    valid_pos = np.nonzero(mask)[0]
    sub_vecs = corpus_emb[valid_pos]
    t0 = time.time()
    idx = hnswlib.Index(space=HNSW_SPACE, dim=sub_vecs.shape[1])
    eff_M = max(2, min(HNSW_M, len(valid_pos) - 1)) if len(valid_pos) > 1 else 2
    idx.init_index(max_elements=max(1, len(sub_vecs)),
                    ef_construction=HNSW_EF_CONSTRUCTION, M=eff_M)
    if len(sub_vecs) > 0:
        idx.add_items(sub_vecs, np.arange(len(sub_vecs)))
        idx.set_ef(min(HNSW_EF_SEARCH, max(K, len(sub_vecs))))
    build_times[name] = time.time() - t0
    built_indexes[name] = (idx, valid_pos)
print("Built", len(built_indexes), "subindexes. Build times (s):", build_times)


Built 3 subindexes. Build times (s): {'base': 0.8532483577728271, 'sel50_label_in_01': 0.3511228561401367, 'sel75_label_ne_3': 0.5540006160736084}


## 13. Serving: for each query filter, pick smallest subsuming subindex or brute-force, per the cost model — not always the subindex

In [13]:
rows = []

for filter_name, (target_pct, mask, ftype) in FILTERS.items():
    actual_pct = 100.0 * mask.sum() / len(mask)
    card_f = int(mask.sum())
    gt_ids, _ = exact_topk_filtered(query_emb, corpus_emb, mask, K)

    # choose smallest built subindex whose filter subsumes this query filter
    subsuming = [(name, hmask) for name, hmask in built_subindexes.items()
                 if subsumes(hmask, mask)]
    subsuming.sort(key=lambda t: int(t[1].sum()))
    chosen_name, chosen_mask = subsuming[0] if subsuming else ("base", built_subindexes["base"])
    card_h = int(chosen_mask.sum())

    est_indexed_cost = cost_to_serve(card_h, card_f)
    est_bf_cost = brute_force_cost(card_f)
    use_indexed = est_indexed_cost <= est_bf_cost

    if use_indexed:
        idx, valid_pos = built_indexes[chosen_name]
        local_mask_within_h = mask[valid_pos]  # which entries in h also satisfy f

        def run():
            lat = []
            ids_all = np.full((len(query_emb), K), -1, dtype=np.int64)
            for qi in range(len(query_emb)):
                t0 = time.perf_counter()
                kk = min(max(K, 5 * K), max(1, len(valid_pos)))
                local_ids, _ = idx.knn_query(query_emb[qi:qi+1], k=kk)
                lat.append(time.perf_counter() - t0)
                global_ids = valid_pos[local_ids[0]]
                filtered = [g for g in global_ids if mask[g]][:K]
                ids_all[qi] = (filtered + [-1] * K)[:K]
            run.last_ids = ids_all
            return lat
        method_used = f"sieve_indexed[{chosen_name}]"
        build_t = build_times.get(chosen_name, 0.0)
    else:
        def run():
            lat = []
            ids_all = np.full((len(query_emb), K), -1, dtype=np.int64)
            for qi in range(len(query_emb)):
                t0 = time.perf_counter()
                ids, _ = exact_topk_filtered(query_emb[qi:qi+1], corpus_emb, mask, K)
                lat.append(time.perf_counter() - t0)
                ids_all[qi] = ids[0]
            run.last_ids = ids_all
            return lat
        method_used = "sieve_bruteforce"
        build_t = 0.0

    lat_stats = time_queries(run)
    recall = recall_at_k(run.last_ids, gt_ids, K)
    rows.append({
        "method": "sieve", "filter": filter_name, "selectivity": round(actual_pct, 3),
        "recall_at_10": recall,
        "mean_latency_ms": lat_stats["mean_latency_ms"],
        "p50_latency_ms": lat_stats["p50_latency_ms"],
        "p95_latency_ms": lat_stats["p95_latency_ms"],
        "qps": lat_stats["qps"],
        "index_build_time_s": build_t,
        "serving_choice": method_used,
        "est_indexed_cost": est_indexed_cost, "est_bf_cost": est_bf_cost,
    })
    print(f"{filter_name:28s} sel={actual_pct:6.2f}%  serving={method_used:28s} "
          f"recall={recall:.3f}")

results_df = pd.DataFrame(rows)
results_df.to_csv("results_sieve.csv", index=False)
print(f"Wrote results_sieve.csv ({len(results_df)} rows)")
results_df


sel100_unfiltered            sel=100.00%  serving=sieve_indexed[base]          recall=0.998
sel75_label_ne_3             sel= 74.63%  serving=sieve_indexed[sel75_label_ne_3] recall=0.999
sel50_label_in_01            sel= 51.17%  serving=sieve_indexed[sel50_label_in_01] recall=0.997
sel25_label_eq_0             sel= 25.60%  serving=sieve_bruteforce             recall=1.000
sel25_label_eq_1             sel= 25.57%  serving=sieve_bruteforce             recall=1.000
sel25_label_eq_2             sel= 23.47%  serving=sieve_bruteforce             recall=1.000
sel25_label_eq_3             sel= 25.37%  serving=sieve_bruteforce             recall=1.000
sel10_label0_tag_lt_0.4      sel=  9.73%  serving=sieve_bruteforce             recall=1.000
sel5_label0_tag_lt_0.2       sel=  5.20%  serving=sieve_bruteforce             recall=1.000
Wrote results_sieve.csv (9 rows)


,method,filter,selectivity,recall_at_10,mean_latency_ms,p50_latency_ms,p95_latency_ms,qps,index_build_time_s,serving_choice,est_indexed_cost,est_bf_cost
0,sieve,sel100_unfiltered,100.000,0.998,0.291433,0.189318,0.409376,3431.315815,0.853248,sieve_indexed[base],800.636757,2072.326584
1,sieve,sel75_label_ne_3,74.633,0.999,0.188622,0.161206,0.218407,5301.609351,0.554001,sieve_indexed[sel75_label_ne_3],743.189375,1546.646407
2,sieve,sel50_label_in_01,51.167,0.997,0.315871,0.147293,2.170542,3165.845742,0.351123,sieve_indexed[sel50_label_in_01],672.228533,1060.340435
3,sieve,sel25_label_eq_0,25.600,1.000,0.640676,0.281933,2.307899,1560.850985,0.000000,sieve_bruteforce,1343.581769,530.515605
4,sieve,sel25_label_eq_1,25.567,1.000,0.663659,0.258573,3.425175,1506.797192,0.000000,sieve_bruteforce,1345.333506,529.824830
5,sieve,sel25_label_eq_2,23.467,1.000,0.595229,0.245202,2.357556,1680.026518,0.000000,sieve_bruteforce,2363.637800,486.305972
6,sieve,sel25_label_eq_3,25.367,1.000,0.621431,0.258909,2.301645,1609.189709,0.000000,sieve_bruteforce,3156.255283,525.680177
7,sieve,sel10_label0_tag_lt_0.4,9.733,1.000,0.199210,0.095301,0.227791,5019.831094,0.000000,sieve_bruteforce,3533.804106,201.706454
8,sieve,sel5_label0_tag_lt_0.2,5.200,1.000,0.165137,0.075227,0.134317,6055.596064,0.000000,sieve_bruteforce,6614.556403,107.760982


## 14. Selected subindex set (inspectable, since candidate space is tiny)

In [14]:
selection_log_df

,added,marginal_ratio,memory_after,cost_after
0,sel50_label_in_01,0.017246,70504.484623,4927.419592
1,sel75_label_ne_3,0.009549,105019.340176,4597.848686
